In [25]:
import os
from datetime import datetime
import numpy as np
import pandas as pd
import xarray as xr
import dask.array as da
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

In [9]:
def get_lat_weights(latitude_values):
    lat_rad = np.deg2rad(latitude_values)
    weights = np.cos(lat_rad)
    return weights / np.mean(weights)

In [10]:
class Normalizer:
    def __init__(self):
        self.mean_in, self.std_in = None, None
        self.mean_out, self.std_out = None, None

    def set_input_statistics(self, mean, std):
        self.mean_in = mean
        self.std_in = std

    def set_output_statistics(self, mean, std):
        self.mean_out = mean
        self.std_out = std

    def normalize(self, data, data_type):
        if data_type == "input":
            return (data - self.mean_in) / self.std_in
        elif data_type == "output":
            return (data - self.mean_out) / self.std_out

    def inverse_transform_output(self, data):
        return data * self.std_out + self.mean_out

In [11]:
class ClimateDataset(Dataset):
    def __init__(self, inputs_dask, outputs_dask, output_is_normalized=True):
        self.size = inputs_dask.shape[0]
        print(f"Creating dataset with {self.size} samples...")

        inputs_np = inputs_dask.compute()
        outputs_np = outputs_dask.compute()

        self.inputs = torch.from_numpy(inputs_np).float()
        self.outputs = torch.from_numpy(outputs_np).float()

        if torch.isnan(self.inputs).any() or torch.isnan(self.outputs).any():
            raise ValueError("NaNs found in dataset")

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        return self.inputs[idx], self.outputs[idx]

In [12]:
path = "processed_data_cse151b_v2_corrupted_ssp245/processed_data_cse151b_v2_corrupted_ssp245.zarr"
input_vars = ["CO2", "SO2", "CH4", "BC", "rsdt"]
output_vars = ["tas", "pr"]
train_ssps = ["ssp126", "ssp370", "ssp585"]
test_ssp = "ssp245"
target_member_id = 0
val_split = 0.1
test_months = 360
batch_size = 32
num_workers = 0
seed = 42
normalizer = Normalizer()

In [13]:
ds = xr.open_zarr(path, consolidated=False, chunks={"time": 24})
spatial_template = ds["rsdt"].isel(time=0, ssp=0, drop=True)
lat = spatial_template.y.values
lon = spatial_template.x.values
area_weights = xr.DataArray(get_lat_weights(lat), dims=["y"], coords={"y": lat})

ds

<xarray.Dataset> Size: 621MB
Dimensions:    (x: 72, y: 48, member_id: 3, ssp: 4, time: 1021, latitude: 48,
                longitude: 72)
Coordinates:
    lat        (x, y) float64 28kB dask.array<chunksize=(72, 48), meta=np.ndarray>
  * member_id  (member_id) int64 24B 0 1 2
    lon        (x, y) float64 28kB dask.array<chunksize=(72, 48), meta=np.ndarray>
  * y          (y) float64 384B -88.59 -84.82 -81.05 ... 81.05 84.82 88.59
  * latitude   (latitude) float64 384B -89.05 -85.26 -81.47 ... 85.26 89.05
  * longitude  (longitude) float64 576B 1.25 6.25 11.25 ... 346.2 351.2 356.2
  * time       (time) object 8kB 2015-01-15 00:00:00 ... 2100-01-15 00:00:00
  * x          (x) float64 576B 1.875 6.875 11.88 16.88 ... 346.9 351.9 356.9
  * ssp        (ssp) <U6 96B 'ssp126' 'ssp245' 'ssp370' 'ssp585'
Data variables:
    pr         (ssp, time, member_id, y, x) float32 169MB dask.array<chunksize=(1, 24, 1, 48, 72), meta=np.ndarray>
    CH4        (ssp, time) float64 33kB dask.array<chunksize=(1, 24), meta=np.ndarray>
    CO2        (ssp, time) float64 33kB dask.array<chunksize=(1, 24), meta=np.ndarray>
    tas        (ssp, time, member_id, y, x) float32 169MB dask.array<chunksize=(1, 24, 1, 48, 72), meta=np.ndarray>
    rsdt       (ssp, time, y, x) float32 56MB dask.array<chunksize=(1, 24, 48, 72), meta=np.ndarray>
    BC         (ssp, time, latitude, longitude) float64 113MB dask.array<chunksize=(1, 24, 48, 72), meta=np.ndarray>
    SO2        (ssp, time, latitude, longitude) float64 113MB dask.array<chunksize=(1, 24, 48, 72), meta=np.ndarray>
Attributes:
    original_member_ids:  ['r10i1p1f1', 'r11i1p1f1', 'r4i1p1f1']
    precipitation_units:  mm/day (converted from kg m-2 s-1)
    source:               CMIP6 data processed for CSE151B
    ssp:                  ssp126

In [14]:
def load_ssp(ssp):
    input_dask, output_dask = [], []
    for var in input_vars:
        da_var = ds[var].sel(ssp=ssp)
        if "latitude" in da_var.dims:
            da_var = da_var.rename({"latitude": "y", "longitude": "x"})
        if "member_id" in da_var.dims:
            da_var = da_var.sel(member_id=target_member_id)
        if set(da_var.dims) == {"time"}:
            da_var = da_var.broadcast_like(spatial_template).transpose("time", "y", "x")
        input_dask.append(da_var.data)

    for var in output_vars:
        da_out = ds[var].sel(ssp=ssp, member_id=target_member_id)
        if "latitude" in da_out.dims:
            da_out = da_out.rename({"latitude": "y", "longitude": "x"})
        output_dask.append(da_out.data)

    return da.stack(input_dask, axis=1), da.stack(output_dask, axis=1)

In [16]:
train_input, train_output, val_input, val_output = [], [], None, None

for ssp in train_ssps:
    x, y = load_ssp(ssp)
    if ssp == "ssp370":
        val_input = x[-test_months:]
        val_output = y[-test_months:]
        train_input.append(x[:-test_months])
        train_output.append(y[:-test_months])
    else:
        train_input.append(x)
        train_output.append(y)
    
train_input = da.concatenate(train_input, axis=0)
train_output = da.concatenate(train_output, axis=0)

normalizer.set_input_statistics(
    mean=da.nanmean(train_input, axis=(0, 2, 3), keepdims=True).compute(),
    std=da.nanstd(train_input, axis=(0, 2, 3), keepdims=True).compute(),
)
normalizer.set_output_statistics(
    mean=da.nanmean(train_output, axis=(0, 2, 3), keepdims=True).compute(),
    std=da.nanstd(train_output, axis=(0, 2, 3), keepdims=True).compute(),
)

train_input_norm = normalizer.normalize(train_input, "input")
train_output_norm = normalizer.normalize(train_output, "output")
val_input_norm = normalizer.normalize(val_input, "input")
val_output_norm = normalizer.normalize(val_output, "output")

test_input, test_output = load_ssp(test_ssp)
test_input = test_input[-test_months:]
test_output = test_output[-test_months:]
test_input_norm = normalizer.normalize(test_input, "input")

In [17]:
train_dataset = ClimateDataset(train_input_norm, train_output_norm)
val_dataset = ClimateDataset(val_input_norm, val_output_norm)
test_dataset = ClimateDataset(test_input_norm, test_output, output_is_normalized=False)

lat = spatial_template.y.values
lon = spatial_template.x.values
area_weights = xr.DataArray(get_lat_weights(lat), dims=["y"], coords={"y": lat})

Creating dataset with 2703 samples...
Creating dataset with 360 samples...
Creating dataset with 360 samples...


In [18]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=True)

val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=True)

test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                             num_workers=num_workers, pin_memory=True)

In [27]:
class MLP(nn.Module):
    def __init__(self, in_channels, out_channels, H, W):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        
        self.in_size = in_channels * H * W
        self.out_size = out_channels * H * W
        
        self.fc1 = nn.Linear(in_channels * H * W, self.in_size // 4)
        self.fc2 = nn.Linear(self.in_size // 4, self.in_size // 8)
        self.fc3 = nn.Linear(self.in_size // 8, self.in_size // 16)
        self.fc4 = nn.Linear(self.in_size // 16, self.out_size // 2)
        self.fc5 = nn.Linear(self.out_size // 2, self.out_size)

    def forward(self, x):
        B, C, H, W = x.shape
        x = x.view(-1, self.in_size)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = F.relu(self.fc4(x))
        return self.fc5(x).view(B, self.out_channels, H, W)

In [28]:
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'Training on device {device}\n')

model = MLP(in_channels=5, out_channels=2, H=48, W=72).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

for epoch in range(15):
    model.train()
    total_train_loss = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        pred = model(xb)
        loss = criterion(pred, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    model.eval()

    with torch.no_grad():
        all_preds = []
        all_targets = []
        total_val_loss = 0
        
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)

            pred = model(xb)
            loss = criterion(pred, yb)
            total_val_loss += loss.item()

            all_preds.append(pred.cpu())
            all_targets.append(yb.cpu())
        
        all_preds = torch.cat(all_preds, dim=0)
        all_targets = torch.cat(all_targets, dim=0)

        all_preds = normalizer.inverse_transform_output(all_preds.detach().cpu().numpy())
        all_targets = normalizer.inverse_transform_output(all_targets.detach().cpu().numpy())
        
        time = np.arange(all_preds.shape[0])

        print(f"Epoch {epoch+1}: Train Loss = {total_train_loss / len(train_loader):.4f}, Valid Loss = {total_val_loss / len(val_loader):.4f}")  
    
        for i, var in enumerate(output_vars):
            p = all_preds[:, i]
            t = all_targets[:, i]
    
            p_xr = xr.DataArray(p, dims=["time", "y", "x"], coords={"time": time, "y": lat, "x": lon})
            t_xr = xr.DataArray(t, dims=["time", "y", "x"], coords={"time": time, "y": lat, "x": lon})
            
            rmse = np.sqrt(((p_xr - t_xr) ** 2).weighted(area_weights).mean(("time", "y", "x")).item())
            mean_rmse = np.sqrt(((p_xr.mean("time") - t_xr.mean("time")) ** 2).weighted(area_weights).mean(("y", "x")).item())
            std_mae = np.abs(p_xr.std("time") - t_xr.std("time")).weighted(area_weights).mean(("y", "x")).item()
    
            print(f"{var}: RMSE={rmse:.4f}, Time-Mean RMSE={mean_rmse:.4f}, Time-Stddev MAE={std_mae:.4f}")

    print('\n')

Training on device cuda

Epoch 1: Train Loss = 0.8611, Valid Loss = 0.3857
tas: RMSE=7.2391, Time-Mean RMSE=5.7687, Time-Stddev MAE=1.9642
pr: RMSE=2.7535, Time-Mean RMSE=0.8921, Time-Stddev MAE=1.5587


Epoch 2: Train Loss = 0.2481, Valid Loss = 0.3571
tas: RMSE=6.4708, Time-Mean RMSE=4.9147, Time-Stddev MAE=1.3835
pr: RMSE=2.7378, Time-Mean RMSE=0.8245, Time-Stddev MAE=1.1916


Epoch 3: Train Loss = 0.2438, Valid Loss = 0.3380
tas: RMSE=5.7285, Time-Mean RMSE=3.9484, Time-Stddev MAE=1.4932
pr: RMSE=2.6897, Time-Mean RMSE=0.7047, Time-Stddev MAE=1.2381


Epoch 4: Train Loss = 0.2398, Valid Loss = 0.3416
tas: RMSE=6.0084, Time-Mean RMSE=4.6684, Time-Stddev MAE=1.2089
pr: RMSE=2.6587, Time-Mean RMSE=0.7174, Time-Stddev MAE=1.1659


Epoch 5: Train Loss = 0.2325, Valid Loss = 0.2942
tas: RMSE=4.7658, Time-Mean RMSE=3.0904, Time-Stddev MAE=0.9391
pr: RMSE=2.5971, Time-Mean RMSE=0.4902, Time-Stddev MAE=1.0968


Epoch 6: Train Loss = 0.2090, Valid Loss = 0.2524
tas: RMSE=3.6272, Time-Mean RM